# ANDES-backed MPM verification workflow

This notebook exercises the ANDES-oriented verification scaffold now hosted directly in `masked-perturbation-model`.
It keeps the demo lightweight: acquire/load a benchmark when available, fall back to a synthetic stable model otherwise, expose read/write channels, and construct the MPM model map bridge.

## Setup

From the repository root, install the package and optional ANDES dependency as needed:

```bash
pip install -e ".[andes,demos]"
```


In [ ]:
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if SRC_ROOT.exists() and str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

PROJECT_ROOT

In [ ]:
from mpm_grid_demo.config import DemoConfig
from mpm_grid_demo.acquire import clone_andes_cases, find_case_files
from mpm_grid_demo.load_case import load_andes_system, setup_power_flow_and_dynamics
from mpm_grid_demo.linearize import (
    extract_andes_state_matrix,
    synthetic_two_area_like_model,
    verify_stability,
)
from mpm_grid_demo.channels import (
    select_speed_reads_and_power_writes,
    select_random_channels,
)
from mpm_grid_demo.bridge import build_lft_from_linear_model

config = DemoConfig(project_root=PROJECT_ROOT)
config

## Acquire a benchmark case when possible

In [ ]:
try:
    case_repo = clone_andes_cases(config)
    case_files = find_case_files(case_repo, keyword=config.preferred_case_keyword)
except Exception as exc:
    print("Could not acquire ANDES cases automatically:", repr(exc))
    case_files = []

print(f"Found {len(case_files)} candidate case files")
case_files[:5]

## Extract a small-signal model or use the synthetic fallback

In [ ]:
linear_model = None

if case_files:
    try:
        system = load_andes_system(case_files[0], setup=False)
        system = setup_power_flow_and_dynamics(system)
        linear_model = extract_andes_state_matrix(system)
        print("Using ANDES-derived linear model")
    except Exception as exc:
        print("ANDES extraction did not complete:", repr(exc))

if linear_model is None and config.use_synthetic_fallback:
    linear_model = synthetic_two_area_like_model()
    print("Using synthetic fallback model")

if linear_model is None:
    raise RuntimeError("No linear model available; enable synthetic fallback or patch ANDES extraction.")

stable, eigvals = verify_stability(linear_model.A, tol=config.stability_tol)
print("source:", linear_model.source)
print("A shape:", linear_model.A.shape)
print("strictly stable:", stable)
print("max real eigenvalue:", float(np.max(np.real(eigvals))))

## Select read/write channels and build the MPM model map

In [ ]:
channels = select_speed_reads_and_power_writes(linear_model.A, linear_model.state_names)
if channels.B_w.shape[1] == 0:
    channels = select_random_channels(
        linear_model.A,
        linear_model.state_names,
        n_channels=min(2, linear_model.A.shape[0] // 2),
        random_seed=config.random_seed,
    )
    print("No power-write states were identified; using random write/read channels for smoke testing.")

print("B_w shape:", channels.B_w.shape)
print("C_r shape:", channels.C_r.shape)
print("D_rw shape:", channels.D_rw.shape)

model_map = build_lft_from_linear_model(linear_model, channels)
print("model-map dimensions:", model_map.ninputs, model_map.noutputs, model_map.nstates)

## Minimal frequency-response smoke test

In [ ]:
omega = 1.0
response = np.asarray(model_map.eval(1j * omega))
print("M(jω) shape:", response.shape)
print("max |entry| at ω=1:", float(np.max(np.abs(response))))